# POI Context Data: EDA and Preprocessing

This notebook prepares the **raw (V1)** and **context-enriched (V2)** city datasets
for the next-POI recommendation experiments.

It performs:

- sequence construction and next-POI target creation;
- duplicate and invalid-record handling;
- categorical cleaning;
- numeric imputation and outlier capping;
- exploratory data analysis;
- export of model-ready Parquet files for Tokyo, Petaling Jaya, and New York City.

## Important evaluation note

This notebook creates **unsplit** processed datasets. Imputation and capping values
are therefore estimated from the full city dataset. This is useful for inspection and
for reproducing the uploaded workflow, but a strict leakage-free experiment should
split the data first and fit all preprocessing statistics on the training portion only.

Set `PROJECT_ROOT` and `DATA_DIR` in the configuration cell, then run the notebook
from top to bottom.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import os
import gc
import json
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ------------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------------
# In Colab, this defaults to a folder in Google Drive.
# Locally, set the POI_PROJECT_ROOT environment variable or edit this path.
DEFAULT_PROJECT_ROOT = (
    "/content/drive/MyDrive/context_trails_data"
    if Path("/content/drive/MyDrive").exists()
    else str(Path.cwd().resolve().parent)
)
PROJECT_ROOT = Path(os.environ.get("POI_PROJECT_ROOT", DEFAULT_PROJECT_ROOT))

# Put the original V1/V2 Parquet files here, or change DATA_DIR.
DATA_DIR = Path(os.environ.get("POI_DATA_DIR", str(PROJECT_ROOT / "input")))

# V2 output folders.
OUTPUT_DIR = PROJECT_ROOT / "preprocessing_outputs" / "V2"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"

# Final datasets are written where the modeling notebook expects them.
PARQUET_ALL_DIR = PROJECT_ROOT / "dataset_versions"
PARQUET_MODEL_DIR = PROJECT_ROOT / "dataset_versions" / "target_valid"

CITY_FILES = {
    "NewYorkCity": "NewYorkCity_V2_context_enriched_LOCAL.parquet",
    "PetalingJaya": "PetalingJaya_V2_context_enriched_LOCAL.parquet",
    "Tokyo": "Tokyo_V2_context_enriched_LOCAL.parquet",
}

SAMPLE_ROWS_FOR_PLOTS = 120_000
RANDOM_STATE = 42

MAKE_PLOTS = True
SAVE_FULL_ALL_ROWS_PARQUET = True
SAVE_MODELING_TARGET_VALID_PARQUET = True
AUTO_DOWNLOAD_ZIP = False

TARGET_COL = "target_next_category_lvlFs"
DROP_UNKNOWN_TARGET_FOR_MODELING = True

UNKNOWN_LABEL = "__UNKNOWN__"
MISSING_LABEL = "__MISSING__"
START_LABEL = "__START_OF_TRAIL__"
NO_HOLIDAY_LABEL = "No holiday"

CAP_Q = 0.99
TRANSIT_IMPUTE_Q = 0.95

CORE_COLS = ["trail_id", "user_id", "venue_id", "timestamp", "city"]
EVENT_DUP_COLS = ["user_id", "trail_id", "timestamp", "venue_id"]

for directory in [
    TABLES_DIR,
    PLOTS_DIR,
    REPORTS_DIR,
    PARQUET_ALL_DIR,
    PARQUET_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input directory:", DATA_DIR)
print("V2 output directory:", OUTPUT_DIR)

## 2. File, output, and type-cleaning helpers

In [ ]:
def find_input_file(filename: str) -> Path:
    """Find a file in /content or common Google Drive locations."""
    candidates = [
        DATA_DIR / filename,
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        Path("/content/drive/MyDrive/context_trails") / filename,
        Path("/content/drive/MyDrive/ContextTrails") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p

    # Recursive fallback. Avoid scanning too deeply if Drive is huge, but this is useful in Colab.
    search_roots = [Path("/content")]
    if Path("/content/drive/MyDrive").exists():
        search_roots.append(Path("/content/drive/MyDrive"))

    for root in search_roots:
        try:
            found = list(root.rglob(filename))
            if found:
                return found[0]
        except Exception:
            pass

    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to /content or set DATA_DIR correctly."
    )


def safe_filename(text: str) -> str:
    return (
        str(text)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("*", "_")
        .replace("?", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )


def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLES_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def save_json(obj, name: str) -> Path:
    path = REPORTS_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    return path


def save_plot(fig, city: str, name: str) -> Path:
    city_plot_dir = PLOTS_DIR / city
    city_plot_dir.mkdir(parents=True, exist_ok=True)
    path = city_plot_dir / f"{name}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


def sample_for_plots(df: pd.DataFrame, n: int = SAMPLE_ROWS_FOR_PLOTS) -> pd.DataFrame:
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=RANDOM_STATE)


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Convert pandas nullable missing values to np.nan where possible."""
    return df.replace({pd.NA: np.nan})


def as_string_clean(s: pd.Series, fill_value: str) -> pd.Series:
    return s.astype("object").where(~s.isna(), fill_value).astype(str)


def bool_to_status(series: pd.Series, true_label="True", false_label="False", missing_label="Unknown") -> pd.Series:
    """Robust conversion of booleans that may be bools, strings, ints, or null."""
    def convert_one(x):
        if pd.isna(x):
            return missing_label
        if isinstance(x, (bool, np.bool_)):
            return true_label if bool(x) else false_label
        sx = str(x).strip().lower()
        if sx in {"true", "1", "yes", "y", "t"}:
            return true_label
        if sx in {"false", "0", "no", "n", "f"}:
            return false_label
        return missing_label
    return series.map(convert_one).astype("object")

## 3. EDA and summary helpers

In [ ]:
def numeric_summary(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    rows = []
    for col in cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "column": col,
            "count": int(s.notna().sum()),
            "missing": int(s.isna().sum()),
            "missing_pct": float(s.isna().mean() * 100),
            "mean": float(s.mean()) if s.notna().any() else np.nan,
            "std": float(s.std()) if s.notna().any() else np.nan,
            "min": float(s.min()) if s.notna().any() else np.nan,
            "p01": float(s.quantile(0.01)) if s.notna().any() else np.nan,
            "p05": float(s.quantile(0.05)) if s.notna().any() else np.nan,
            "p25": float(s.quantile(0.25)) if s.notna().any() else np.nan,
            "median": float(s.median()) if s.notna().any() else np.nan,
            "p75": float(s.quantile(0.75)) if s.notna().any() else np.nan,
            "p95": float(s.quantile(0.95)) if s.notna().any() else np.nan,
            "p99": float(s.quantile(0.99)) if s.notna().any() else np.nan,
            "max": float(s.max()) if s.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def value_counts_table(df: pd.DataFrame, col: str, city: str, top_n: int = 30) -> pd.DataFrame:
    if col not in df.columns:
        return pd.DataFrame()
    vc = df[col].astype("object").where(~df[col].isna(), MISSING_LABEL).value_counts(dropna=False).head(top_n)
    out = vc.rename_axis(col).reset_index(name="count")
    out["pct"] = out["count"] / len(df) * 100 if len(df) else 0
    out.insert(0, "city", city)
    return out


def bar_counts_plot(counts: pd.Series, title: str, xlabel: str = "Count", ylabel: str = "", max_items: int = 20):
    counts = counts.head(max_items).sort_values()
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(counts) + 1.5)))
    ax.barh(counts.index.astype(str), counts.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    return fig


def simple_hist_plot(series: pd.Series, title: str, xlabel: str, bins: int = 50):
    s = pd.to_numeric(series, errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(9, 5))
    if len(s) > 0:
        ax.hist(s, bins=bins)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    return fig


def line_counts_plot(counts: pd.Series, title: str, xlabel: str):
    counts = counts.sort_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(counts.index.astype(str), counts.values, marker="o")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    return fig


def make_missingness_table(df: pd.DataFrame, city: str, stage: str) -> pd.DataFrame:
    n = len(df)
    rows = []
    for col in df.columns:
        miss = int(df[col].isna().sum())
        rows.append({
            "city": city,
            "stage": stage,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_count": miss,
            "missing_pct": (miss / n * 100) if n else 0,
            "n_unique_including_missing": int(df[col].nunique(dropna=False)),
        })
    return pd.DataFrame(rows).sort_values(["missing_pct", "missing_count"], ascending=False)


def get_existing_cols(df: pd.DataFrame, cols: list) -> list:
    return [c for c in cols if c in df.columns]

## 4. Imputation, capping, and feature-role helpers

In [ ]:
def fill_numeric_by_group_then_global(df, col, group_col, new_col=None, flag_col=None):
    if new_col is None:
        new_col = f"{col}_imputed"
    if flag_col is None:
        flag_col = f"{col}_was_imputed"

    x = pd.to_numeric(df[col], errors="coerce")
    df[flag_col] = x.isna()

    if group_col in df.columns:
        group_median = x.groupby(df[group_col]).transform("median")
    else:
        group_median = pd.Series(np.nan, index=df.index)

    global_median = x.median()
    if pd.isna(global_median):
        global_median = 0.0

    df[new_col] = x.fillna(group_median).fillna(global_median)
    return {
        "source_col": col,
        "output_col": new_col,
        "flag_col": flag_col,
        "method": f"{group_col} median -> global median",
        "global_fallback": float(global_median),
        "n_imputed": int(df[flag_col].sum()),
        "pct_imputed": float(df[flag_col].mean() * 100) if len(df) else 0,
    }


def fill_numeric_by_month_hour_then_global(df, col, new_col=None, flag_col=None):
    if new_col is None:
        new_col = f"{col}_imputed"
    if flag_col is None:
        flag_col = f"{col}_was_imputed"

    x = pd.to_numeric(df[col], errors="coerce")
    df[flag_col] = x.isna()

    if "month" in df.columns and "hour" in df.columns:
        group_median = x.groupby([df["month"], df["hour"]]).transform("median")
    else:
        group_median = pd.Series(np.nan, index=df.index)

    global_median = x.median()
    if pd.isna(global_median):
        global_median = 0.0

    df[new_col] = x.fillna(group_median).fillna(global_median)
    return {
        "source_col": col,
        "output_col": new_col,
        "flag_col": flag_col,
        "method": "month+hour median -> global median",
        "global_fallback": float(global_median),
        "n_imputed": int(df[flag_col].sum()),
        "pct_imputed": float(df[flag_col].mean() * 100) if len(df) else 0,
    }


def cap_numeric_column(df, source_col, q=CAP_Q, output_col=None, flag_col=None, lower=None):
    if output_col is None:
        output_col = f"{source_col}_capped"
    if flag_col is None:
        flag_col = f"{source_col}_was_capped"

    x = pd.to_numeric(df[source_col], errors="coerce")
    upper = x.quantile(q)
    if pd.isna(upper):
        upper = x.max()
    if pd.isna(upper):
        upper = 0.0

    capped = x.copy()
    if lower is not None:
        capped = capped.clip(lower=lower)
    capped = capped.clip(upper=upper)

    df[output_col] = capped
    df[flag_col] = x.notna() & (x > upper)

    return {
        "source_col": source_col,
        "output_col": output_col,
        "flag_col": flag_col,
        "method": f"cap upper at q={q}",
        "upper_cap": float(upper),
        "n_capped": int(df[flag_col].sum()),
        "pct_capped": float(df[flag_col].mean() * 100) if len(df) else 0,
    }


def build_feature_role_report(df: pd.DataFrame) -> pd.DataFrame:
    roles = []
    for col in df.columns:
        role = "raw_or_context_feature"
        if col.endswith("_was_imputed") or col.endswith("_was_capped") or col.endswith("_missing") or col.endswith("_invalid") or col.endswith("_extreme"):
            role = "preprocessing_flag"
        elif col.endswith("_imputed"):
            role = "imputed_numeric_feature"
        elif col.endswith("_capped"):
            role = "capped_numeric_feature"
        elif col.endswith("_clean") or col in {"opening_status"}:
            role = "cleaned_categorical_or_cleaned_raw"
        elif col.startswith("target_next_"):
            role = "target"
        elif col.startswith("prev_") or col in {"step_index", "trail_length", "minutes_since_prev", "minutes_until_next", "is_first_in_trail", "is_last_in_trail"}:
            role = "sequence_feature"
        elif col in {"trail_id", "user_id", "venue_id", "fsq_id", "qid", "timestamp", "date"}:
            role = "identifier_or_raw_time_do_not_onehot_directly"
        roles.append({"column": col, "dtype": str(df[col].dtype), "role": role})
    return pd.DataFrame(roles)

## 5. Sequence construction

In [ ]:
def add_sequence_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create previous/next target features on the full city data before any row sampling."""
    df = df.copy()
    df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    # Use user_id + trail_id to be safer if trail IDs are not globally unique.
    group_cols = [c for c in ["user_id", "trail_id"] if c in df.columns]
    if not group_cols:
        group_cols = ["trail_id"] if "trail_id" in df.columns else []

    sort_cols = group_cols + ["timestamp_parsed"]
    df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    if group_cols:
        g = df.groupby(group_cols, sort=False, dropna=False)

        for c in ["venue_id", "category", "category_lvlFs"]:
            if c in df.columns:
                df[f"prev_{c}"] = g[c].shift(1)
                df[f"target_next_{c}"] = g[c].shift(-1)

        df["step_index"] = g.cumcount()
        df["trail_length"] = g["venue_id"].transform("size") if "venue_id" in df.columns else g.size()
        df["is_first_in_trail"] = df["step_index"].eq(0)
        df["is_last_in_trail"] = df["step_index"].eq(df["trail_length"] - 1)

        df["minutes_since_prev"] = g["timestamp_parsed"].diff().dt.total_seconds() / 60.0
        df["minutes_until_next"] = (g["timestamp_parsed"].shift(-1) - df["timestamp_parsed"]).dt.total_seconds() / 60.0
    else:
        df["step_index"] = np.arange(len(df))
        df["trail_length"] = len(df)
        df["is_first_in_trail"] = False
        df["is_last_in_trail"] = False
        df["minutes_since_prev"] = np.nan
        df["minutes_until_next"] = np.nan

    return df

## 6. Cleaning and imputation

In [ ]:
def clean_and_impute_unsplit(df: pd.DataFrame, city: str):
    """
    Clean and impute one full city dataset without splitting.
    Returns:
        df: processed dataframe
        reports: dict with removal/imputation/capping/action summaries
    """
    reports = {
        "city": city,
        "initial_rows": int(len(df)),
        "initial_cols": int(df.shape[1]),
        "removals": [],
        "imputations": [],
        "caps": [],
        "categorical_cleaning": [],
    }

    df = normalize_missing_values(df)

    # Add sequence features before removals and before any sampling.
    df = add_sequence_features(df)

    # Remove rows missing core fields.
    before = len(df)
    existing_core = get_existing_cols(df, CORE_COLS)
    if existing_core:
        mask_core_ok = df[existing_core].notna().all(axis=1)
        df = df.loc[mask_core_ok].copy()
    after = len(df)
    reports["removals"].append({
        "step": "remove_missing_core_fields",
        "columns": existing_core,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after,
        "reason": "Core ID/time/city fields are required for valid visits.",
    })

    # Remove exact duplicate rows.
    before = len(df)
    df = df.drop_duplicates().copy()
    after = len(df)
    reports["removals"].append({
        "step": "remove_exact_duplicate_rows",
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after,
        "reason": "Exact duplicate rows add no information.",
    })

    # Remove duplicated event rows.
    before = len(df)
    event_cols = get_existing_cols(df, EVENT_DUP_COLS)
    if event_cols:
        df = df.drop_duplicates(subset=event_cols, keep="first").copy()
    after = len(df)
    reports["removals"].append({
        "step": "remove_duplicate_user_trail_timestamp_venue",
        "columns": event_cols,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after,
        "reason": "Duplicate visit event records can distort sequence counts.",
    })

    # Clean coordinate problems.
    lat = pd.to_numeric(df["latitude"], errors="coerce") if "latitude" in df.columns else pd.Series(np.nan, index=df.index)
    lon = pd.to_numeric(df["longitude"], errors="coerce") if "longitude" in df.columns else pd.Series(np.nan, index=df.index)

    coordinate_issue = df["coordinate_issue"].astype("object") if "coordinate_issue" in df.columns else pd.Series("", index=df.index)
    has_valid_coordinates = df["has_valid_coordinates"] if "has_valid_coordinates" in df.columns else pd.Series(True, index=df.index)

    valid_bool_status = bool_to_status(has_valid_coordinates, true_label="True", false_label="False", missing_label="Unknown")
    bad_coord = (
        lat.isna() | lon.isna()
        | lat.lt(-90) | lat.gt(90)
        | lon.lt(-180) | lon.gt(180)
        | ((lat == -1) & (lon == -1))
        | coordinate_issue.astype(str).str.contains("sentinel|missing|invalid", case=False, na=False)
        | valid_bool_status.eq("False")
    )

    df["coordinate_missing_or_invalid"] = bad_coord
    df["latitude_clean"] = lat.where(~bad_coord, np.nan)
    df["longitude_clean"] = lon.where(~bad_coord, np.nan)

    reports["categorical_cleaning"].append({
        "step": "clean_coordinates",
        "new_columns": ["coordinate_missing_or_invalid", "latitude_clean", "longitude_clean"],
        "n_flagged": int(bad_coord.sum()),
        "pct_flagged": float(bad_coord.mean() * 100) if len(df) else 0,
        "reason": "Sentinel/invalid coordinates are set to missing but the row is preserved.",
    })

    # Clean category columns.
    for col in ["category", "category_lvlFs"]:
        if col in df.columns:
            clean_col = f"{col}_clean"
            flag_col = f"{col}_was_missing_or_unknown"
            s = df[col].astype("object")
            missing_or_unknown = s.isna() | s.astype(str).str.strip().str.lower().isin(
                {"", "nan", "none", "null", "unknown", "__unknown__"}
            )
            df[clean_col] = s.where(~missing_or_unknown, UNKNOWN_LABEL).astype(str)
            df[flag_col] = missing_or_unknown
            reports["categorical_cleaning"].append({
                "step": f"clean_{col}",
                "source_col": col,
                "output_col": clean_col,
                "flag_col": flag_col,
                "n_flagged": int(missing_or_unknown.sum()),
                "pct_flagged": float(missing_or_unknown.mean() * 100) if len(df) else 0,
                "reason": "Unknown input category is preserved as an explicit category.",
            })

    # Clean previous categorical sequence features.
    for col in ["prev_venue_id", "prev_category", "prev_category_lvlFs"]:
        if col in df.columns:
            clean_col = f"{col}_clean"
            flag_col = f"{col}_was_imputed"
            missing = df[col].isna()
            df[clean_col] = df[col].astype("object").where(~missing, START_LABEL).astype(str)
            df[flag_col] = missing
            reports["imputations"].append({
                "source_col": col,
                "output_col": clean_col,
                "flag_col": flag_col,
                "method": f"missing previous value -> {START_LABEL}",
                "n_imputed": int(missing.sum()),
                "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
            })

    # Clean target labels for easy filtering later. We do not impute target.
    for col in ["target_next_category", "target_next_category_lvlFs", "target_next_venue_id"]:
        if col in df.columns:
            clean_col = f"{col}_clean"
            s = df[col].astype("object")
            missing_or_unknown = s.isna() | s.astype(str).str.strip().str.lower().isin(
                {"", "nan", "none", "null", "unknown", "__unknown__"}
            )
            df[clean_col] = s.where(~missing_or_unknown, np.nan)

    # Holiday cleaning.
    if "holiday_name" in df.columns:
        missing = df["holiday_name"].isna()
        df["holiday_name_clean"] = df["holiday_name"].astype("object").where(~missing, NO_HOLIDAY_LABEL).astype(str)
        df["holiday_name_was_imputed"] = missing
        reports["imputations"].append({
            "source_col": "holiday_name",
            "output_col": "holiday_name_clean",
            "flag_col": "holiday_name_was_imputed",
            "method": f"missing -> {NO_HOLIDAY_LABEL}",
            "n_imputed": int(missing.sum()),
            "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
        })

    if "holiday_is_public_holiday" in df.columns:
        missing = df["holiday_is_public_holiday"].isna()
        status = bool_to_status(df["holiday_is_public_holiday"], true_label="True", false_label="False", missing_label="False")
        df["holiday_is_public_holiday_clean"] = status.eq("True")
        df["holiday_is_public_holiday_was_imputed"] = missing
        reports["imputations"].append({
            "source_col": "holiday_is_public_holiday",
            "output_col": "holiday_is_public_holiday_clean",
            "flag_col": "holiday_is_public_holiday_was_imputed",
            "method": "missing -> False",
            "n_imputed": int(missing.sum()),
            "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
        })

    # Opening hours cleaning.
    if "is_open_when_visited" in df.columns:
        missing = df["is_open_when_visited"].isna()
        df["opening_status"] = bool_to_status(
            df["is_open_when_visited"],
            true_label="Open",
            false_label="Closed",
            missing_label="Unknown",
        )
        df["opening_status_was_missing"] = missing
        reports["categorical_cleaning"].append({
            "step": "clean_opening_status",
            "source_col": "is_open_when_visited",
            "output_col": "opening_status",
            "flag_col": "opening_status_was_missing",
            "n_flagged": int(missing.sum()),
            "pct_flagged": float(missing.mean() * 100) if len(df) else 0,
            "reason": "Missing opening hours mean unknown, not closed.",
        })

    if "opening_column_used" in df.columns:
        missing = df["opening_column_used"].isna()
        df["opening_column_used_clean"] = df["opening_column_used"].astype("object").where(~missing, MISSING_LABEL).astype(str)
        df["opening_column_used_was_missing"] = missing

    # Weather categorical cleaning.
    if "conditions" in df.columns:
        missing = df["conditions"].isna()
        df["conditions_clean"] = df["conditions"].astype("object").where(~missing, MISSING_LABEL).astype(str)
        df["conditions_was_missing"] = missing

    if "preciptype" in df.columns:
        precip = pd.to_numeric(df["precip"], errors="coerce") if "precip" in df.columns else pd.Series(np.nan, index=df.index)
        missing = df["preciptype"].isna()
        fill_none = missing & precip.fillna(0).le(0)
        fill_missing = missing & ~fill_none
        df["preciptype_clean"] = df["preciptype"].astype("object")
        df.loc[fill_none, "preciptype_clean"] = "none"
        df.loc[fill_missing, "preciptype_clean"] = MISSING_LABEL
        df["preciptype_was_imputed"] = missing
        reports["imputations"].append({
            "source_col": "preciptype",
            "output_col": "preciptype_clean",
            "flag_col": "preciptype_was_imputed",
            "method": "missing and precip<=0 -> none; otherwise __MISSING__",
            "n_imputed": int(missing.sum()),
            "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
        })

    # Numeric POI metadata imputation by category.
    group_col = "category_lvlFs_clean" if "category_lvlFs_clean" in df.columns else None
    for col in ["price", "rating", "total_ratings", "total_tips"]:
        if col in df.columns:
            reports["imputations"].append(
                fill_numeric_by_group_then_global(df, col, group_col, new_col=f"{col}_imputed", flag_col=f"{col}_was_imputed")
            )

    # Derived popularity after imputation.
    if "total_ratings_imputed" in df.columns:
        df["log1p_total_ratings_imputed"] = np.log1p(pd.to_numeric(df["total_ratings_imputed"], errors="coerce").clip(lower=0))
    if "total_tips_imputed" in df.columns:
        df["log1p_total_tips_imputed"] = np.log1p(pd.to_numeric(df["total_tips_imputed"], errors="coerce").clip(lower=0))

    # Weather numeric imputation.
    for col in ["temp", "precip", "windspeed"]:
        if col in df.columns:
            reports["imputations"].append(
                fill_numeric_by_month_hour_then_global(df, col, new_col=f"{col}_imputed", flag_col=f"{col}_was_imputed")
            )

    # Holiday distance numeric imputation.
    for col in ["holiday_days_since_previous", "holiday_days_until_next"]:
        if col in df.columns:
            reports["imputations"].append(
                fill_numeric_by_group_then_global(df, col, "month", new_col=f"{col}_imputed", flag_col=f"{col}_was_imputed")
            )

    # Transit nearest stop distance: treat invalid coordinates as unknown before imputation.
    if "public_transport_nearest_stop_km" in df.columns:
        source = pd.to_numeric(df["public_transport_nearest_stop_km"], errors="coerce")
        source = source.where(~df["coordinate_missing_or_invalid"], np.nan)
        source = source.where(source.ge(0), np.nan)

        flag_col = "public_transport_nearest_stop_km_was_imputed"
        df[flag_col] = source.isna()

        impute_value = source.quantile(TRANSIT_IMPUTE_Q)
        if pd.isna(impute_value):
            impute_value = source.median()
        if pd.isna(impute_value):
            impute_value = 0.0

        df["public_transport_nearest_stop_km_imputed"] = source.fillna(impute_value)

        reports["imputations"].append({
            "source_col": "public_transport_nearest_stop_km",
            "output_col": "public_transport_nearest_stop_km_imputed",
            "flag_col": flag_col,
            "method": f"invalid coordinates/missing -> city q{TRANSIT_IMPUTE_Q}",
            "fallback_value": float(impute_value),
            "n_imputed": int(df[flag_col].sum()),
            "pct_imputed": float(df[flag_col].mean() * 100) if len(df) else 0,
        })

    # Transit categorical cleaning.
    for col in ["public_transport_nearest_stop_feed", "public_transport_nearest_stop_mode"]:
        if col in df.columns:
            missing = df[col].isna()
            df[f"{col}_clean"] = df[col].astype("object").where(~missing, MISSING_LABEL).astype(str)
            df[f"{col}_was_missing"] = missing

    # Transit boolean cleaning.
    for col in [
        "public_transport_has_stop_250m",
        "public_transport_has_stop_500m",
        "public_transport_has_stop_1000m",
    ]:
        if col in df.columns:
            df[f"{col}_status"] = bool_to_status(df[col], true_label="True", false_label="False", missing_label="Unknown")
            df[f"{col}_was_missing"] = df[col].isna()

    # Transit counts: fill missing as 0 but keep flag. This is practical, but the flag prevents hiding missingness.
    transport_count_cols = [
        "public_transport_stop_count_250m",
        "public_transport_bus_stop_count_250m",
        "public_transport_rail_stop_count_250m",
        "public_transport_subway_stop_count_250m",
        "public_transport_metro_stop_count_250m",
        "public_transport_stop_count_500m",
        "public_transport_bus_stop_count_500m",
        "public_transport_rail_stop_count_500m",
        "public_transport_subway_stop_count_500m",
        "public_transport_metro_stop_count_500m",
        "public_transport_stop_count_1000m",
        "public_transport_bus_stop_count_1000m",
        "public_transport_rail_stop_count_1000m",
        "public_transport_subway_stop_count_1000m",
        "public_transport_metro_stop_count_1000m",
    ]
    for col in get_existing_cols(df, transport_count_cols):
        x = pd.to_numeric(df[col], errors="coerce")
        missing = x.isna()
        df[f"{col}_imputed"] = x.fillna(0).clip(lower=0)
        df[f"{col}_was_imputed"] = missing
        df[f"log1p_{col}_imputed"] = np.log1p(df[f"{col}_imputed"])
        reports["imputations"].append({
            "source_col": col,
            "output_col": f"{col}_imputed",
            "flag_col": f"{col}_was_imputed",
            "method": "missing -> 0 with missing flag; negative clipped to 0",
            "n_imputed": int(missing.sum()),
            "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
        })

    # Time gap cleaning.
    for col in ["minutes_since_prev", "minutes_until_next"]:
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            invalid = x.lt(0)
            x = x.where(~invalid, np.nan)
            missing = x.isna()
            if col == "minutes_since_prev":
                # First stop has no previous event. 0 is interpretable, with flag.
                fallback = 0.0
            else:
                # Last stop has no next event. For all-row file keep it flagged.
                fallback = x.median()
                if pd.isna(fallback):
                    fallback = 0.0
            df[f"{col}_imputed"] = x.fillna(fallback)
            df[f"{col}_was_imputed"] = missing
            df[f"log1p_{col}_imputed"] = np.log1p(pd.to_numeric(df[f"{col}_imputed"], errors="coerce").clip(lower=0))
            reports["imputations"].append({
                "source_col": col,
                "output_col": f"{col}_imputed",
                "flag_col": f"{col}_was_imputed",
                "method": f"negative -> missing; missing -> {fallback}",
                "fallback_value": float(fallback),
                "n_imputed": int(missing.sum()),
                "pct_imputed": float(missing.mean() * 100) if len(df) else 0,
            })

    # Capping outliers.
    for source_col, out_col in [
        ("precip_imputed", "precip_capped"),
        ("public_transport_nearest_stop_km_imputed", "public_transport_nearest_stop_km_capped"),
        ("log1p_total_ratings_imputed", "log1p_total_ratings_capped"),
        ("log1p_total_tips_imputed", "log1p_total_tips_capped"),
        ("log1p_minutes_since_prev_imputed", "log1p_minutes_since_prev_capped"),
    ]:
        if source_col in df.columns:
            reports["caps"].append(
                cap_numeric_column(
                    df,
                    source_col=source_col,
                    q=CAP_Q,
                    output_col=out_col,
                    flag_col=f"{out_col}_was_capped",
                    lower=0 if "log1p" in source_col or "precip" in source_col or "transport" in source_col else None,
                )
            )

    # Basic date/time fallback cleaning. We keep original columns too.
    if "visit_time_bucket" in df.columns:
        missing = df["visit_time_bucket"].isna()
        df["visit_time_bucket_clean"] = df["visit_time_bucket"].astype("object").where(~missing, MISSING_LABEL).astype(str)
        df["visit_time_bucket_was_missing"] = missing

    reports["final_rows_all_rows_file"] = int(len(df))
    reports["final_cols_all_rows_file"] = int(df.shape[1])

    return df, reports

## 7. EDA reports and plots

In [ ]:
def create_city_summaries_and_plots(df_original: pd.DataFrame, df_processed: pd.DataFrame, city: str):
    """Save EDA summaries and plots for one city."""
    created = {"tables": [], "plots": []}

    # Schema and missingness.
    schema_rows = []
    for col in df_original.columns:
        schema_rows.append({
            "city": city,
            "column": col,
            "dtype": str(df_original[col].dtype),
            "n_unique_including_missing": int(df_original[col].nunique(dropna=False)),
            "missing_count": int(df_original[col].isna().sum()),
            "missing_pct": float(df_original[col].isna().mean() * 100),
        })
    created["tables"].append(str(save_table(pd.DataFrame(schema_rows), f"{city}_01_original_schema_missingness")))

    created["tables"].append(str(save_table(make_missingness_table(df_processed, city, "processed"), f"{city}_02_processed_missingness")))

    # Core quality.
    q = {
        "city": city,
        "rows_original": int(len(df_original)),
        "cols_original": int(df_original.shape[1]),
        "rows_processed": int(len(df_processed)),
        "cols_processed": int(df_processed.shape[1]),
    }
    for col in CORE_COLS:
        if col in df_original.columns:
            q[f"missing_{col}_original"] = int(df_original[col].isna().sum())

    if set(EVENT_DUP_COLS).issubset(df_original.columns):
        q["duplicate_event_rows_original"] = int(df_original.duplicated(subset=EVENT_DUP_COLS).sum())
    q["exact_duplicate_rows_original"] = int(df_original.duplicated().sum())

    if "coordinate_missing_or_invalid" in df_processed.columns:
        q["coordinate_missing_or_invalid_processed"] = int(df_processed["coordinate_missing_or_invalid"].sum())
        q["coordinate_missing_or_invalid_pct_processed"] = float(df_processed["coordinate_missing_or_invalid"].mean() * 100)

    created["tables"].append(str(save_table(pd.DataFrame([q]), f"{city}_03_core_quality_summary")))

    # Numeric summaries.
    raw_numeric_cols = get_existing_cols(df_original, [
        "price", "rating", "total_ratings", "total_tips",
        "temp", "precip", "windspeed",
        "latitude", "longitude",
        "holiday_days_since_previous", "holiday_days_until_next",
        "public_transport_nearest_stop_km",
        "public_transport_stop_count_250m",
        "public_transport_stop_count_500m",
        "public_transport_stop_count_1000m",
    ])
    created["tables"].append(str(save_table(numeric_summary(df_original, raw_numeric_cols), f"{city}_04_original_numeric_summary")))

    processed_numeric_cols = get_existing_cols(df_processed, [
        "price_imputed", "rating_imputed", "total_ratings_imputed", "total_tips_imputed",
        "log1p_total_ratings_capped", "log1p_total_tips_capped",
        "temp_imputed", "precip_imputed", "precip_capped", "windspeed_imputed",
        "public_transport_nearest_stop_km_imputed", "public_transport_nearest_stop_km_capped",
        "minutes_since_prev_imputed", "log1p_minutes_since_prev_capped",
        "trail_length", "step_index",
    ])
    created["tables"].append(str(save_table(numeric_summary(df_processed, processed_numeric_cols), f"{city}_05_processed_numeric_summary")))

    # Value counts tables.
    for col in [
        "category_lvlFs", "category", "category_lvlFs_clean", "target_next_category_lvlFs_clean",
        "visit_time_bucket", "visit_time_bucket_clean", "opening_status", "conditions_clean",
        "preciptype_clean", "holiday_name_clean", "public_transport_nearest_stop_mode_clean",
    ]:
        if col in df_processed.columns:
            tbl = value_counts_table(df_processed, col, city, top_n=50)
            if not tbl.empty:
                created["tables"].append(str(save_table(tbl, f"{city}_06_value_counts_{safe_filename(col)}")))

    # Target availability summary.
    target_summary = []
    for col in ["target_next_venue_id", "target_next_category", "target_next_category_lvlFs", "target_next_category_lvlFs_clean"]:
        if col in df_processed.columns:
            s = df_processed[col]
            target_summary.append({
                "city": city,
                "target_col": col,
                "available_count": int(s.notna().sum()),
                "available_pct": float(s.notna().mean() * 100),
                "missing_count": int(s.isna().sum()),
                "unique_available": int(s.nunique(dropna=True)),
            })
    created["tables"].append(str(save_table(pd.DataFrame(target_summary), f"{city}_07_target_availability")))

    # Imputation and capping summaries.
    flag_cols = [c for c in df_processed.columns if c.endswith("_was_imputed") or c.endswith("_was_missing")]
    flag_summary = []
    for c in flag_cols:
        s = df_processed[c].fillna(False).astype(bool)
        flag_summary.append({
            "city": city,
            "flag_col": c,
            "count_true": int(s.sum()),
            "pct_true": float(s.mean() * 100) if len(s) else 0,
        })
    created["tables"].append(str(save_table(pd.DataFrame(flag_summary).sort_values("pct_true", ascending=False), f"{city}_08_imputation_missing_flags_summary")))

    cap_cols = [c for c in df_processed.columns if c.endswith("_was_capped")]
    cap_summary = []
    for c in cap_cols:
        s = df_processed[c].fillna(False).astype(bool)
        cap_summary.append({
            "city": city,
            "flag_col": c,
            "count_true": int(s.sum()),
            "pct_true": float(s.mean() * 100) if len(s) else 0,
        })
    created["tables"].append(str(save_table(pd.DataFrame(cap_summary).sort_values("pct_true", ascending=False), f"{city}_09_capping_flags_summary")))

    # Sequence summary by trails.
    if "trail_id" in df_processed.columns:
        trail_summary = df_processed.groupby(["user_id", "trail_id"], dropna=False).agg(
            n_rows=("venue_id", "size"),
            min_time=("timestamp_parsed", "min"),
            max_time=("timestamp_parsed", "max"),
        ).reset_index()
        trail_summary["duration_minutes"] = (trail_summary["max_time"] - trail_summary["min_time"]).dt.total_seconds() / 60.0
        created["tables"].append(str(save_table(numeric_summary(trail_summary, ["n_rows", "duration_minutes"]), f"{city}_10_trail_numeric_summary")))

    # Feature role report.
    created["tables"].append(str(save_table(build_feature_role_report(df_processed), f"{city}_11_feature_role_report")))

    if not MAKE_PLOTS:
        return created

    # Use sample for plotting.
    plot_df = sample_for_plots(df_processed)

    # Missingness plot from original.
    miss_orig = make_missingness_table(df_original, city, "original").set_index("column")["missing_pct"].sort_values(ascending=False)
    miss_orig = miss_orig[miss_orig > 0]
    if len(miss_orig) > 0:
        fig = bar_counts_plot(miss_orig.head(25), f"{city}: top original missingness (%)", xlabel="Missing percent", max_items=25)
        created["plots"].append(str(save_plot(fig, city, "01_top_original_missingness_pct")))

    # Category and target plots.
    for col, title in [
        ("category_lvlFs_clean", "Top current broad categories"),
        ("category_clean", "Top current categories"),
        ("target_next_category_lvlFs_clean", "Top next broad-category targets"),
        ("prev_category_lvlFs_clean", "Top previous broad categories"),
    ]:
        if col in plot_df.columns:
            counts = plot_df[col].astype("object").where(~plot_df[col].isna(), MISSING_LABEL).value_counts().head(20)
            fig = bar_counts_plot(counts, f"{city}: {title}", max_items=20)
            created["plots"].append(str(save_plot(fig, city, f"02_{safe_filename(col)}_top_counts")))

    # Time plots.
    if "hour" in plot_df.columns:
        counts = plot_df["hour"].value_counts().sort_index()
        fig = line_counts_plot(counts, f"{city}: visits by hour", "Hour")
        created["plots"].append(str(save_plot(fig, city, "03_visits_by_hour")))

    if "dayofweek" in plot_df.columns:
        counts = plot_df["dayofweek"].value_counts().sort_index()
        fig = line_counts_plot(counts, f"{city}: visits by day of week", "Day of week")
        created["plots"].append(str(save_plot(fig, city, "04_visits_by_dayofweek")))

    if "month" in plot_df.columns:
        counts = plot_df["month"].value_counts().sort_index()
        fig = line_counts_plot(counts, f"{city}: visits by month", "Month")
        created["plots"].append(str(save_plot(fig, city, "05_visits_by_month")))

    if "visit_time_bucket_clean" in plot_df.columns:
        counts = plot_df["visit_time_bucket_clean"].value_counts()
        fig = bar_counts_plot(counts, f"{city}: visit time bucket", max_items=20)
        created["plots"].append(str(save_plot(fig, city, "06_visit_time_bucket")))

    # Opening, holiday, weather categorical plots.
    for col in ["opening_status", "holiday_name_clean", "preciptype_clean", "conditions_clean", "public_transport_nearest_stop_mode_clean"]:
        if col in plot_df.columns:
            counts = plot_df[col].astype("object").where(~plot_df[col].isna(), MISSING_LABEL).value_counts().head(20)
            fig = bar_counts_plot(counts, f"{city}: {col}", max_items=20)
            created["plots"].append(str(save_plot(fig, city, f"07_{safe_filename(col)}")))

    # Numeric histograms.
    for col, label in [
        ("price_imputed", "Price imputed"),
        ("rating_imputed", "Rating imputed"),
        ("log1p_total_ratings_capped", "Log1p total ratings capped"),
        ("log1p_total_tips_capped", "Log1p total tips capped"),
        ("temp_imputed", "Temperature imputed"),
        ("precip_capped", "Precipitation capped"),
        ("windspeed_imputed", "Wind speed imputed"),
        ("public_transport_nearest_stop_km_capped", "Nearest transport stop km capped"),
        ("log1p_minutes_since_prev_capped", "Log1p minutes since previous capped"),
        ("trail_length", "Trail length"),
        ("step_index", "Step index"),
    ]:
        if col in plot_df.columns:
            fig = simple_hist_plot(plot_df[col], f"{city}: {label}", label)
            created["plots"].append(str(save_plot(fig, city, f"08_hist_{safe_filename(col)}")))

    # Imputation/capping summary plots.
    if flag_summary:
        flag_s = pd.DataFrame(flag_summary).set_index("flag_col")["pct_true"].sort_values(ascending=False)
        flag_s = flag_s[flag_s > 0]
        if len(flag_s) > 0:
            fig = bar_counts_plot(flag_s.head(25), f"{city}: imputation/missing flags (%)", xlabel="Percent flagged", max_items=25)
            created["plots"].append(str(save_plot(fig, city, "09_imputation_missing_flags_pct")))

    if cap_summary:
        cap_s = pd.DataFrame(cap_summary).set_index("flag_col")["pct_true"].sort_values(ascending=False)
        cap_s = cap_s[cap_s > 0]
        if len(cap_s) > 0:
            fig = bar_counts_plot(cap_s.head(20), f"{city}: capping flags (%)", xlabel="Percent capped", max_items=20)
            created["plots"].append(str(save_plot(fig, city, "10_capping_flags_pct")))

    # Simple relationship plots on sample.
    if {"rating_imputed", "log1p_total_ratings_capped"}.issubset(plot_df.columns):
        tmp = plot_df[["rating_imputed", "log1p_total_ratings_capped"]].dropna()
        if len(tmp) > 0:
            tmp = tmp.sample(n=min(len(tmp), 30000), random_state=RANDOM_STATE)
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.scatter(tmp["rating_imputed"], tmp["log1p_total_ratings_capped"], s=4, alpha=0.25)
            ax.set_title(f"{city}: rating vs log popularity")
            ax.set_xlabel("Rating imputed")
            ax.set_ylabel("Log1p total ratings capped")
            created["plots"].append(str(save_plot(fig, city, "11_scatter_rating_vs_log_popularity")))

    if {"public_transport_nearest_stop_km_capped", "rating_imputed"}.issubset(plot_df.columns):
        tmp = plot_df[["public_transport_nearest_stop_km_capped", "rating_imputed"]].dropna()
        if len(tmp) > 0:
            tmp = tmp.sample(n=min(len(tmp), 30000), random_state=RANDOM_STATE)
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.scatter(tmp["public_transport_nearest_stop_km_capped"], tmp["rating_imputed"], s=4, alpha=0.25)
            ax.set_title(f"{city}: transit distance vs rating")
            ax.set_xlabel("Nearest stop km capped")
            ax.set_ylabel("Rating imputed")
            created["plots"].append(str(save_plot(fig, city, "12_scatter_transit_distance_vs_rating")))

    return created

## 8. Process one enriched (V2) city

In [ ]:
def process_one_city(city: str, filename: str):
    print(f"\n=== Processing {city} ===")
    file_path = find_input_file(filename)
    print("Input:", file_path)

    df_original = pd.read_parquet(file_path)
    df_original["city"] = df_original["city"].fillna(city) if "city" in df_original.columns else city

    original_rows = len(df_original)
    original_cols = df_original.shape[1]
    print(f"Loaded {city}: {original_rows:,} rows, {original_cols:,} columns")

    # Clean and impute.
    df_processed, report = clean_and_impute_unsplit(df_original, city)

    # Create modeling target-valid file by filtering target after sequence construction.
    modeling_df = None
    if TARGET_COL in df_processed.columns or f"{TARGET_COL}_clean" in df_processed.columns:
        target_clean = f"{TARGET_COL}_clean" if f"{TARGET_COL}_clean" in df_processed.columns else TARGET_COL
        valid_target = df_processed[target_clean].notna()

        if DROP_UNKNOWN_TARGET_FOR_MODELING:
            valid_target &= ~df_processed[target_clean].astype(str).str.strip().str.lower().isin(
                {"", "unknown", "__unknown__", "nan", "none", "null"}
            )

        modeling_df = df_processed.loc[valid_target].copy()
        modeling_df["modeling_target"] = modeling_df[target_clean].astype(str)
        report["target_clean_col_used_for_modeling"] = target_clean
        report["modeling_rows_target_valid"] = int(len(modeling_df))
        report["modeling_rows_removed_due_to_missing_or_unknown_target"] = int(len(df_processed) - len(modeling_df))
        report["modeling_target_unique_classes"] = int(modeling_df["modeling_target"].nunique(dropna=True))
    else:
        report["target_clean_col_used_for_modeling"] = None
        report["modeling_rows_target_valid"] = 0
        report["modeling_rows_removed_due_to_missing_or_unknown_target"] = None
        report["modeling_target_unique_classes"] = 0

    # Save EDA outputs.
    created = create_city_summaries_and_plots(df_original, df_processed, city)
    report["created_tables_count"] = len(created["tables"])
    report["created_plots_count"] = len(created["plots"])

    # Save parquet outputs.
    if SAVE_FULL_ALL_ROWS_PARQUET:
        all_path = PARQUET_ALL_DIR / f"{city}_V2_preprocessed_unsplit_all_rows.parquet"
        df_processed.to_parquet(all_path, index=False)
        report["saved_all_rows_parquet"] = str(all_path)

    if SAVE_MODELING_TARGET_VALID_PARQUET and modeling_df is not None:
        model_path = PARQUET_MODEL_DIR / f"{city}_V2_preprocessed_unsplit_modeling_target_valid.parquet"
        modeling_df.to_parquet(model_path, index=False)
        report["saved_modeling_target_valid_parquet"] = str(model_path)

    # Save report.
    save_json(report, f"{city}_full_city_preprocessing_report")

    print(f"Processed rows all-row file: {len(df_processed):,}")
    if modeling_df is not None:
        print(f"Target-valid modeling rows: {len(modeling_df):,}")
    print(f"Tables: {len(created['tables'])}, plots: {len(created['plots'])}")

    # Compact summary rows for global reports.
    removal_summary = []
    for r in report.get("removals", []):
        row = {"city": city}
        row.update(r)
        removal_summary.append(row)

    imputation_summary = []
    for r in report.get("imputations", []):
        row = {"city": city}
        row.update(r)
        imputation_summary.append(row)

    capping_summary = []
    for r in report.get("caps", []):
        row = {"city": city}
        row.update(r)
        capping_summary.append(row)

    action_summary = []
    for r in report.get("categorical_cleaning", []):
        row = {"city": city}
        row.update(r)
        action_summary.append(row)

    city_summary = {
        "city": city,
        "input_file": str(file_path),
        "original_rows": int(original_rows),
        "original_cols": int(original_cols),
        "processed_all_rows": int(len(df_processed)),
        "processed_cols": int(df_processed.shape[1]),
        "modeling_target_valid_rows": int(len(modeling_df)) if modeling_df is not None else 0,
        "modeling_target_unique_classes": int(modeling_df["modeling_target"].nunique(dropna=True)) if modeling_df is not None and "modeling_target" in modeling_df.columns else 0,
        "tables_created": len(created["tables"]),
        "plots_created": len(created["plots"]),
    }

    # Free memory.
    del df_original, df_processed
    if modeling_df is not None:
        del modeling_df
    gc.collect()

    return city_summary, removal_summary, imputation_summary, capping_summary, action_summary

## 9. Run enriched (V2) preprocessing for all cities

In [ ]:
# ============================================================
# Run all cities
# ============================================================

all_city_summaries = []
all_removals = []
all_imputations = []
all_caps = []
all_actions = []

for city, filename in CITY_FILES.items():
    city_summary, removal_summary, imputation_summary, capping_summary, action_summary = process_one_city(city, filename)
    all_city_summaries.append(city_summary)
    all_removals.extend(removal_summary)
    all_imputations.extend(imputation_summary)
    all_caps.extend(capping_summary)
    all_actions.extend(action_summary)

save_table(pd.DataFrame(all_city_summaries), "ALL_CITIES_00_processing_summary")
save_table(pd.DataFrame(all_removals), "ALL_CITIES_01_removal_summary")
save_table(pd.DataFrame(all_imputations), "ALL_CITIES_02_imputation_summary")
save_table(pd.DataFrame(all_caps), "ALL_CITIES_03_capping_summary")
save_table(pd.DataFrame(all_actions), "ALL_CITIES_04_categorical_cleaning_summary")

# Combined README.
readme = f"""
# Context Trails V2 combined EDA + unsplit preprocessing

This output was created by the combined V2 notebook.

## What it does

- Uses only V2 context-enriched files.
- Processes one city at a time to reduce RAM use.
- Creates EDA tables and PNG plots.
- Creates sequence features on the full city data before sampling.
- Creates unsplit cleaned/imputed parquet files.
- Adds explicit flags for imputation, missingness, capping, and coordinate issues.
- Saves both:
  - all-row preprocessed unsplit parquet files
  - target-valid unsplit modeling parquet files

## Important modeling note

This is an unsplit preprocessing pipeline. Imputation and capping statistics are learned from the full city data.
This is useful for inspection and a clean full dataset. For strict final model evaluation,
use a split-first pipeline where imputation/capping are learned from train only.

## Target

Prepared target: `{TARGET_COL}`

The modeling parquet files contain `modeling_target`, based on `{TARGET_COL}_clean` when available.

## Important output folders

- `processed_unsplit_all_rows/`
- `processed_unsplit_modeling_target_valid/`
- `tables/`
- `plots/`
- `reports/`

## Key flags added

Examples:
- `price_was_imputed`
- `rating_was_imputed`
- `total_ratings_was_imputed`
- `total_tips_was_imputed`
- `precip_was_imputed`
- `precip_capped_was_capped`
- `coordinate_missing_or_invalid`
- `opening_status_was_missing`
- `holiday_name_was_imputed`
- `public_transport_nearest_stop_km_was_imputed`
- `public_transport_nearest_stop_km_capped_was_capped`
"""

readme_path = OUTPUT_DIR / "README_combined_EDA_unsplit_preprocessing.md"
readme_path.write_text(readme, encoding="utf-8")

print("\nFinished all cities.")
print("Output directory:", OUTPUT_DIR)

## 10. Archive enriched preprocessing outputs

In [ ]:
# ============================================================
# Zip outputs and download
# ============================================================

zip_path = OUTPUT_DIR.parent / "context_trails_V2_preprocessing_outputs.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUTPUT_DIR.parent))

print("Created zip:", zip_path)

if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    files.download(str(zip_path))

# Raw baseline (V1) preprocessing

The following section reuses the same cleaning functions for the V1 baseline files.
Weather, holiday, opening-hours, and public-transport columns are absent in V1 and
are skipped automatically.

## 11. Configure raw (V1) processing

In [ ]:
from pathlib import Path
import gc
import json
import zipfile
import pandas as pd

# ------------------------------------------------------------
# Safety check: make sure the original V2 helper functions exist
# ------------------------------------------------------------
required_functions = [
    "find_input_file",
    "clean_and_impute_unsplit",
    "create_city_summaries_and_plots",
    "save_table",
    "save_json",
]

missing_functions = [f for f in required_functions if f not in globals()]
if missing_functions:
    raise RuntimeError(
        "You need to run the setup/helper/function cells from the V2 notebook first. "
        f"Missing functions: {missing_functions}"
    )

# ------------------------------------------------------------
# V1 configuration
# ------------------------------------------------------------
VERSION_LABEL = "V1"

# Keep same DATA_DIR as before unless you want to change it.
# DATA_DIR = Path("/content")

OUTPUT_DIR = PROJECT_ROOT / "preprocessing_outputs" / "V1"

CITY_FILES_V1 = {
    "NewYorkCity": "NewYorkCity_V1_baseline_no_context.parquet",
    "PetalingJaya": "PetalingJaya_V1_baseline_no_context.parquet",
    "Tokyo": "Tokyo_V1_baseline_no_context.parquet",
}

# Output controls
MAKE_PLOTS = True
SAVE_FULL_ALL_ROWS_PARQUET = True
SAVE_MODELING_TARGET_VALID_PARQUET = True
AUTO_DOWNLOAD_ZIP = False

# Same first modeling target as V2.
# This target is created by add_sequence_features() from category_lvlFs.
TARGET_COL = "target_next_category_lvlFs"
DROP_UNKNOWN_TARGET_FOR_MODELING = True

# Create new V1 output folders.
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"
PARQUET_ALL_DIR = PROJECT_ROOT / "dataset_versions"
PARQUET_MODEL_DIR = PROJECT_ROOT / "dataset_versions" / "target_valid"

for d in [TABLES_DIR, PLOTS_DIR, REPORTS_DIR, PARQUET_ALL_DIR, PARQUET_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("V1 output directory:", OUTPUT_DIR)



## 12. Process one raw (V1) city

In [ ]:
def process_one_city_v1(city: str, filename: str):
    print(f"\n=== Processing {city} {VERSION_LABEL} ===")

    file_path = find_input_file(filename)
    print("Input:", file_path)

    df_original = pd.read_parquet(file_path)

    if "city" in df_original.columns:
        df_original["city"] = df_original["city"].fillna(city)
    else:
        df_original["city"] = city

    original_rows = len(df_original)
    original_cols = df_original.shape[1]

    print(f"Loaded {city} {VERSION_LABEL}: {original_rows:,} rows, {original_cols:,} columns")

    # Reuse the same preprocessing logic.
    # Since V1 lacks V2-only columns, those parts are skipped by the function.
    df_processed, report = clean_and_impute_unsplit(df_original, city)

    report["version"] = VERSION_LABEL
    report["input_file"] = str(file_path)
    report["note"] = (
        "V1 baseline preprocessing. V2-only context columns such as weather, holidays, "
        "opening-hours, and public transport are absent and therefore skipped."
    )

    # Create target-valid modeling dataframe.
    modeling_df = None
    target_clean = f"{TARGET_COL}_clean" if f"{TARGET_COL}_clean" in df_processed.columns else TARGET_COL

    if target_clean in df_processed.columns:
        valid_target = df_processed[target_clean].notna()

        if DROP_UNKNOWN_TARGET_FOR_MODELING:
            valid_target &= ~df_processed[target_clean].astype(str).str.strip().str.lower().isin(
                {"", "unknown", "__unknown__", "nan", "none", "null"}
            )

        modeling_df = df_processed.loc[valid_target].copy()
        modeling_df["modeling_target"] = modeling_df[target_clean].astype(str)

        report["target_clean_col_used_for_modeling"] = target_clean
        report["modeling_rows_target_valid"] = int(len(modeling_df))
        report["modeling_rows_removed_due_to_missing_or_unknown_target"] = int(len(df_processed) - len(modeling_df))
        report["modeling_target_unique_classes"] = int(modeling_df["modeling_target"].nunique(dropna=True))
    else:
        report["target_clean_col_used_for_modeling"] = None
        report["modeling_rows_target_valid"] = 0
        report["modeling_rows_removed_due_to_missing_or_unknown_target"] = None
        report["modeling_target_unique_classes"] = 0

    # Save EDA summaries and plots.
    created = create_city_summaries_and_plots(df_original, df_processed, city)
    report["created_tables_count"] = len(created["tables"])
    report["created_plots_count"] = len(created["plots"])

    # Save processed unsplit all-row parquet.
    if SAVE_FULL_ALL_ROWS_PARQUET:
        all_path = PARQUET_ALL_DIR / f"{city}_{VERSION_LABEL}_preprocessed_unsplit_all_rows.parquet"
        df_processed.to_parquet(all_path, index=False)
        report["saved_all_rows_parquet"] = str(all_path)

    # Save target-valid modeling parquet.
    if SAVE_MODELING_TARGET_VALID_PARQUET and modeling_df is not None:
        model_path = PARQUET_MODEL_DIR / f"{city}_{VERSION_LABEL}_preprocessed_unsplit_modeling_target_valid.parquet"
        modeling_df.to_parquet(model_path, index=False)
        report["saved_modeling_target_valid_parquet"] = str(model_path)

    # Save city report.
    save_json(report, f"{city}_{VERSION_LABEL}_full_city_preprocessing_report")

    print(f"Processed all-row file rows: {len(df_processed):,}")
    if modeling_df is not None:
        print(f"Target-valid modeling rows: {len(modeling_df):,}")
    print(f"Tables: {len(created['tables'])}, plots: {len(created['plots'])}")

    # Compact rows for global reports.
    removal_summary = []
    for r in report.get("removals", []):
        row = {"city": city, "version": VERSION_LABEL}
        row.update(r)
        removal_summary.append(row)

    imputation_summary = []
    for r in report.get("imputations", []):
        row = {"city": city, "version": VERSION_LABEL}
        row.update(r)
        imputation_summary.append(row)

    capping_summary = []
    for r in report.get("caps", []):
        row = {"city": city, "version": VERSION_LABEL}
        row.update(r)
        capping_summary.append(row)

    action_summary = []
    for r in report.get("categorical_cleaning", []):
        row = {"city": city, "version": VERSION_LABEL}
        row.update(r)
        action_summary.append(row)

    city_summary = {
        "city": city,
        "version": VERSION_LABEL,
        "input_file": str(file_path),
        "original_rows": int(original_rows),
        "original_cols": int(original_cols),
        "processed_all_rows": int(len(df_processed)),
        "processed_cols": int(df_processed.shape[1]),
        "modeling_target_valid_rows": int(len(modeling_df)) if modeling_df is not None else 0,
        "modeling_target_unique_classes": (
            int(modeling_df["modeling_target"].nunique(dropna=True))
            if modeling_df is not None and "modeling_target" in modeling_df.columns
            else 0
        ),
        "tables_created": len(created["tables"]),
        "plots_created": len(created["plots"]),
    }

    del df_original, df_processed
    if modeling_df is not None:
        del modeling_df
    gc.collect()

    return city_summary, removal_summary, imputation_summary, capping_summary, action_summary

## 13. Run raw (V1) preprocessing for all cities

In [ ]:
# ------------------------------------------------------------
# Run V1 preprocessing for all cities
# ------------------------------------------------------------
all_city_summaries_v1 = []
all_removals_v1 = []
all_imputations_v1 = []
all_caps_v1 = []
all_actions_v1 = []

for city, filename in CITY_FILES_V1.items():
    city_summary, removal_summary, imputation_summary, capping_summary, action_summary = process_one_city_v1(city, filename)

    all_city_summaries_v1.append(city_summary)
    all_removals_v1.extend(removal_summary)
    all_imputations_v1.extend(imputation_summary)
    all_caps_v1.extend(capping_summary)
    all_actions_v1.extend(action_summary)

save_table(pd.DataFrame(all_city_summaries_v1), "ALL_CITIES_V1_00_processing_summary")
save_table(pd.DataFrame(all_removals_v1), "ALL_CITIES_V1_01_removal_summary")
save_table(pd.DataFrame(all_imputations_v1), "ALL_CITIES_V1_02_imputation_summary")
save_table(pd.DataFrame(all_caps_v1), "ALL_CITIES_V1_03_capping_summary")
save_table(pd.DataFrame(all_actions_v1), "ALL_CITIES_V1_04_categorical_cleaning_summary")

## 14. Document and archive raw preprocessing outputs

In [ ]:
# ------------------------------------------------------------
# Save README
# ------------------------------------------------------------
readme = f"""
# Context Trails V1 combined EDA + unsplit preprocessing

This output was created by the V1 add-on cell.

## What it does

- Uses only V1 baseline files.
- Processes one city at a time to reduce RAM use.
- Reuses the same shared preprocessing logic as the V2 pipeline.
- Creates sequence features on the full city data before sampling.
- Creates unsplit cleaned/imputed parquet files.
- Adds explicit flags for imputation, missingness, capping, and coordinate issues.
- Saves EDA tables and PNG plots.
- Saves both:
  - all-row preprocessed unsplit parquet files
  - target-valid unsplit modeling parquet files

## Important difference from V2

V1 does not contain rich context columns such as:

- weather
- holidays
- opening-hours context
- public transport context

Therefore, those steps are skipped automatically. The V1 output is useful as a baseline representation.

## Modeling target

Prepared target: `{TARGET_COL}`

The modeling parquet files contain:

- `modeling_target`

which is based on `{TARGET_COL}_clean` when available.

## Modeling note

This is an unsplit preprocessing pipeline. Imputation and capping statistics are learned from the full city data.
For strict final model evaluation, use a split-first pipeline where imputation/capping are learned from train only.
"""

readme_path = OUTPUT_DIR / "README_V1_combined_EDA_unsplit_preprocessing.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("Saved README:", readme_path)


# ------------------------------------------------------------
# Zip V1 outputs and optionally download
# ------------------------------------------------------------
zip_path = OUTPUT_DIR.parent / "context_trails_V1_preprocessing_outputs.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUTPUT_DIR.parent))

print("Created V1 zip:", zip_path)

if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    files.download(str(zip_path))